# Experiment: Wildfire Perimeters and VIIRS Exploratory Analysis

Objective:
- Explore wildfire perimeter records used by the pipeline and quantify their temporal/spatial overlap with VIIRS FRP points.
- Verify VIIRS start/end coverage against wildfire windows and inspect how well per-fire and per-sample windows are populated.
- Capture concrete findings that inform follow-up modeling notebooks (PM2.5 analysis will be handled separately).


In [ ]:
# Setup: imports and reproducibility
from __future__ import annotations

from datetime import timedelta
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from gribcheck.config import load_config
from gribcheck.fire import load_filtered_fire_records

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 160)
plt.style.use('seaborn-v0_8-whitegrid')

SEED = 7
np.random.seed(SEED)
SEED


## Plan

- Load configuration and confirm paths for wildfire perimeters, VIIRS cache, and wildfire sample index artifacts.
- Profile wildfire perimeter records after pipeline filters (type/date/size/start-end validity).
- Profile VIIRS FRP cache (date range, hourly density, FRP distribution).
- Compare wildfire and VIIRS temporal windows, with specific focus on start/end coverage.
- Quantify fire-level VIIRS overlap in active windows using wildfire bounding boxes.
- Quantify sample-level VIIRS overlap for daily source hours (`0,6,12,18`) from wildfire index rows.
- Summarize findings and next actions.


In [ ]:
# Load config and define reusable helpers
PROJECT_ROOT = Path.cwd()
CONFIG_PATH = PROJECT_ROOT / 'config/pipeline_config.yaml'
if not CONFIG_PATH.exists():
    raise FileNotFoundError(f'Config file not found: {CONFIG_PATH}')

cfg = load_config(CONFIG_PATH)

def _dedupe_index_df(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if 'sample_id' in out.columns:
        out = out.sort_values('sample_id').drop_duplicates(subset=['sample_id'], keep='last')
    if 'sample_key' in out.columns:
        out = out.drop_duplicates(subset=['sample_key'], keep='last')
    return out.reset_index(drop=True)

def load_wildfire_index_df(index_parquet_path: Path) -> pd.DataFrame:
    if index_parquet_path.exists():
        return _dedupe_index_df(pd.read_parquet(index_parquet_path))
    checkpoint_path = index_parquet_path.with_suffix('.checkpoint.jsonl')
    if checkpoint_path.exists():
        return _dedupe_index_df(pd.read_json(checkpoint_path, lines=True))
    raise FileNotFoundError(
        f'No wildfire index found at {index_parquet_path} or {checkpoint_path}'
    )

def parse_source_hours(text: object) -> list[int]:
    if text is None or (isinstance(text, float) and np.isnan(text)):
        return []
    raw = str(text).strip()
    if not raw:
        return []
    out = []
    for part in raw.split(','):
        p = part.strip()
        if p:
            out.append(int(p))
    return sorted(set(out))

path_table = pd.DataFrame(
    [
        {'artifact': 'wildfire_perimeter_geojson', 'path': str(cfg.paths.wildfire_perimeter_geojson), 'exists': cfg.paths.wildfire_perimeter_geojson.exists()},
        {'artifact': 'viirs_cache_parquet', 'path': str(cfg.viirs.cache_parquet), 'exists': cfg.viirs.cache_parquet.exists()},
        {'artifact': 'wildfire_index_parquet', 'path': str(cfg.paths.wildfire_index_output), 'exists': cfg.paths.wildfire_index_output.exists()},
        {'artifact': 'wildfire_index_checkpoint', 'path': str(cfg.paths.wildfire_index_output.with_suffix('.checkpoint.jsonl')), 'exists': cfg.paths.wildfire_index_output.with_suffix('.checkpoint.jsonl').exists()},
    ]
)

path_table


In [ ]:
# Load pipeline-filtered wildfire records and build a DataFrame
fire_records = load_filtered_fire_records(cfg)

fire_df = pd.DataFrame(
    [
        {
            'fire_id': rec.unique_fire_id,
            'incident_name': rec.incident_name,
            'incident_type': rec.incident_type,
            'state': rec.state,
            'start_time_utc': rec.start_time_utc,
            'end_time_utc': rec.end_time_utc,
            'start_date': rec.start_date,
            'end_date': rec.end_date,
            'size_acres': rec.size_acres,
            'min_lon': rec.min_lon,
            'min_lat': rec.min_lat,
            'max_lon': rec.max_lon,
            'max_lat': rec.max_lat,
        }
        for rec in fire_records
    ]
)

if fire_df.empty:
    raise RuntimeError('No wildfire records were loaded after filtering.')

fire_df['start_time_utc'] = pd.to_datetime(fire_df['start_time_utc'], utc=True)
fire_df['end_time_utc'] = pd.to_datetime(fire_df['end_time_utc'], utc=True)
fire_df['start_hour_utc'] = fire_df['start_time_utc'].dt.floor('h')
fire_df['end_hour_utc'] = fire_df['end_time_utc'].dt.floor('h')
fire_df['duration_hours'] = ((fire_df['end_hour_utc'] - fire_df['start_hour_utc']).dt.total_seconds() / 3600.0 + 1).clip(lower=0)
fire_df['duration_days'] = fire_df['duration_hours'] / 24.0
fire_df['start_year'] = fire_df['start_time_utc'].dt.year
fire_df['end_year'] = fire_df['end_time_utc'].dt.year

fire_overview = pd.Series(
    {
        'fire_count': int(len(fire_df)),
        'start_min_utc': fire_df['start_time_utc'].min(),
        'start_max_utc': fire_df['start_time_utc'].max(),
        'end_min_utc': fire_df['end_time_utc'].min(),
        'end_max_utc': fire_df['end_time_utc'].max(),
        'median_size_acres': float(fire_df['size_acres'].median()),
        'median_duration_days': float(fire_df['duration_days'].median()),
    },
    name='value',
)

fire_overview.to_frame()


In [ ]:
# Wildfire perimeter exploratory tables
size_quantiles = fire_df['size_acres'].quantile([0.1, 0.25, 0.5, 0.75, 0.9, 0.99]).rename('size_acres')
duration_quantiles = fire_df['duration_days'].quantile([0.1, 0.25, 0.5, 0.75, 0.9, 0.99]).rename('duration_days')

states_top = (
    fire_df.groupby('state', as_index=False)
    .agg(
        fires=('fire_id', 'nunique'),
        median_size_acres=('size_acres', 'median'),
        median_duration_days=('duration_days', 'median'),
    )
    .sort_values('fires', ascending=False)
    .head(15)
    .reset_index(drop=True)
)

starts_by_year = (
    fire_df.groupby('start_year', as_index=False)
    .agg(fires=('fire_id', 'nunique'), median_size_acres=('size_acres', 'median'))
    .sort_values('start_year')
    .reset_index(drop=True)
)

print('Top states by filtered fire count')
states_top


In [ ]:
# Wildfire perimeter visuals
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

starts_by_year.plot(x='start_year', y='fires', kind='bar', ax=axes[0, 0], color='#1f77b4', legend=False)
axes[0, 0].set_title('Filtered Wildfire Count by Start Year')
axes[0, 0].set_xlabel('Year')
axes[0, 0].set_ylabel('Fire count')

fire_df['size_acres'].plot(kind='hist', bins=40, ax=axes[0, 1], color='#ff7f0e')
axes[0, 1].set_title('Fire Size Distribution (acres)')
axes[0, 1].set_xlabel('Acres')

fire_df['duration_days'].plot(kind='hist', bins=40, ax=axes[1, 0], color='#2ca02c')
axes[1, 0].set_title('Fire Duration Distribution (days)')
axes[1, 0].set_xlabel('Days')

top_states_counts = states_top[['state', 'fires']].sort_values('fires', ascending=True)
axes[1, 1].barh(top_states_counts['state'], top_states_counts['fires'], color='#9467bd')
axes[1, 1].set_title('Top States by Fire Count (filtered)')
axes[1, 1].set_xlabel('Fire count')

plt.tight_layout()
plt.show()

pd.concat([size_quantiles, duration_quantiles], axis=1)


## VIIRS FRP Cache Exploration

The wildfire pipeline stores VIIRS FRP points as hourly points in UTC. This section checks volume, time coverage, and FRP intensity characteristics.


In [ ]:
# Load VIIRS hourly points cache
viirs_path = cfg.viirs.cache_parquet
if not viirs_path.exists():
    raise FileNotFoundError(f'VIIRS cache not found: {viirs_path}')

viirs_df = pd.read_parquet(viirs_path)
expected_cols = {'latitude', 'longitude', 'frp', 'hour_utc'}
missing_cols = expected_cols.difference(viirs_df.columns)
if missing_cols:
    raise ValueError(f'VIIRS cache missing columns: {sorted(missing_cols)}')

viirs_df['hour_utc'] = pd.to_datetime(viirs_df['hour_utc'], utc=True)
viirs_df['date_utc'] = viirs_df['hour_utc'].dt.date
viirs_df['year_utc'] = viirs_df['hour_utc'].dt.year

viirs_overview = pd.Series(
    {
        'viirs_row_count': int(len(viirs_df)),
        'hour_min_utc': viirs_df['hour_utc'].min(),
        'hour_max_utc': viirs_df['hour_utc'].max(),
        'unique_hours': int(viirs_df['hour_utc'].nunique()),
        'median_frp': float(viirs_df['frp'].median()),
        'frp_p90': float(viirs_df['frp'].quantile(0.9)),
        'frp_p99': float(viirs_df['frp'].quantile(0.99)),
    },
    name='value',
)

viirs_overview.to_frame()


In [ ]:
# VIIRS visuals and compact tables
viirs_daily_counts = (
    viirs_df.groupby('date_utc', as_index=False)
    .agg(points=('frp', 'size'), total_frp=('frp', 'sum'))
    .sort_values('date_utc')
    .reset_index(drop=True)
)

viirs_yearly = (
    viirs_df.groupby('year_utc', as_index=False)
    .agg(points=('frp', 'size'), median_frp=('frp', 'median'), frp_p95=('frp', lambda s: float(s.quantile(0.95))))
    .sort_values('year_utc')
    .reset_index(drop=True)
)

fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))

axes[0].plot(pd.to_datetime(viirs_daily_counts['date_utc']), viirs_daily_counts['points'], linewidth=1.0, color='#1f77b4')
axes[0].set_title('VIIRS Daily Point Count')
axes[0].set_xlabel('Date (UTC)')
axes[0].set_ylabel('Points')

axes[1].hist(np.log1p(viirs_df['frp']), bins=60, color='#ff7f0e')
axes[1].set_title('log1p(FRP) Distribution')
axes[1].set_xlabel('log1p(FRP)')

viirs_yearly.plot(x='year_utc', y='points', kind='bar', ax=axes[2], color='#2ca02c', legend=False)
axes[2].set_title('VIIRS Points by Year')
axes[2].set_xlabel('Year')
axes[2].set_ylabel('Points')

plt.tight_layout()
plt.show()

viirs_yearly


## Temporal Alignment: Wildfire Start/End Windows vs VIIRS Coverage

Focus question: do wildfire windows (start to end) fit inside the available VIIRS time range, and where are the edge gaps?


In [ ]:
# Compare wildfire temporal windows against VIIRS start/end availability
viirs_min_hour = viirs_df['hour_utc'].min()
viirs_max_hour = viirs_df['hour_utc'].max()

fire_df['window_fully_covered_by_viirs'] = (
    (fire_df['start_hour_utc'] >= viirs_min_hour)
    & (fire_df['end_hour_utc'] <= viirs_max_hour)
)

fire_df['temporal_overlap_start'] = fire_df['start_hour_utc'].apply(lambda ts: max(ts, viirs_min_hour))
fire_df['temporal_overlap_end'] = fire_df['end_hour_utc'].apply(lambda ts: min(ts, viirs_max_hour))
fire_df['viirs_overlap_hours'] = (
    ((fire_df['temporal_overlap_end'] - fire_df['temporal_overlap_start']).dt.total_seconds() // 3600) + 1
).clip(lower=0).astype(int)

fire_df['viirs_overlap_fraction'] = (
    fire_df['viirs_overlap_hours'] / fire_df['duration_hours'].clip(lower=1)
)

fire_df['hours_from_viirs_start_to_fire_start'] = (
    (fire_df['start_hour_utc'] - viirs_min_hour).dt.total_seconds() / 3600.0
)
fire_df['hours_from_fire_end_to_viirs_end'] = (
    (viirs_max_hour - fire_df['end_hour_utc']).dt.total_seconds() / 3600.0
)

temporal_summary = pd.Series(
    {
        'viirs_min_hour_utc': viirs_min_hour,
        'viirs_max_hour_utc': viirs_max_hour,
        'fires_total': int(len(fire_df)),
        'fires_fully_inside_viirs_window': int(fire_df['window_fully_covered_by_viirs'].sum()),
        'fires_partially_or_not_covered': int((~fire_df['window_fully_covered_by_viirs']).sum()),
        'median_viirs_overlap_fraction': float(fire_df['viirs_overlap_fraction'].median()),
        'p10_viirs_overlap_fraction': float(fire_df['viirs_overlap_fraction'].quantile(0.10)),
    },
    name='value',
)

temporal_summary.to_frame()


In [ ]:
# Temporal alignment diagnostics around start/end edges
edge_cases = fire_df.loc[
    ~fire_df['window_fully_covered_by_viirs'],
    [
        'fire_id',
        'incident_name',
        'state',
        'start_time_utc',
        'end_time_utc',
        'viirs_overlap_fraction',
        'hours_from_viirs_start_to_fire_start',
        'hours_from_fire_end_to_viirs_end',
    ],
].sort_values(['start_time_utc', 'fire_id'])

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

axes[0].hist(fire_df['viirs_overlap_fraction'], bins=30, color='#1f77b4')
axes[0].set_title('Fire Window Fraction Covered by VIIRS')
axes[0].set_xlabel('Coverage fraction')

axes[1].scatter(
    fire_df['hours_from_viirs_start_to_fire_start'],
    fire_df['hours_from_fire_end_to_viirs_end'],
    s=18,
    alpha=0.6,
    color='#ff7f0e',
)
axes[1].axvline(0, color='black', linestyle='--', linewidth=1)
axes[1].axhline(0, color='black', linestyle='--', linewidth=1)
axes[1].set_title('Start/End Margins vs VIIRS Range (hours)')
axes[1].set_xlabel('Fire start minus VIIRS start')
axes[1].set_ylabel('VIIRS end minus fire end')

plt.tight_layout()
plt.show()

edge_cases.head(20)


## Fire-Level Spatial + Temporal Match to VIIRS

Match rule used here: a VIIRS point is considered matched when it falls inside the fire bounding box and within the fire active window (`start_hour_utc` to `end_hour_utc`).

Note: this uses bounding boxes (same fields used by the wildfire raster index), not exact polygon containment.


In [ ]:
# Compute VIIRS overlap with each fire's bounding box during active window
viirs_by_day = {
    day: grp[['hour_utc', 'latitude', 'longitude', 'frp']]
    for day, grp in viirs_df.groupby('date_utc', sort=False)
}

overlap_rows = []
for row in fire_df.itertuples(index=False):
    current_day = row.start_hour_utc.date()
    end_day = row.end_hour_utc.date()

    point_count = 0
    total_frp = 0.0
    max_frp = np.nan
    matched_hours: set[pd.Timestamp] = set()
    first_match = None
    last_match = None

    while current_day <= end_day:
        day_df = viirs_by_day.get(current_day)
        if day_df is not None and not day_df.empty:
            in_bbox = (
                day_df['longitude'].between(row.min_lon, row.max_lon)
                & day_df['latitude'].between(row.min_lat, row.max_lat)
            )
            subset = day_df.loc[in_bbox]
            if not subset.empty:
                subset = subset.loc[
                    (subset['hour_utc'] >= row.start_hour_utc)
                    & (subset['hour_utc'] <= row.end_hour_utc)
                ]
                if not subset.empty:
                    point_count += int(len(subset))
                    total_frp += float(subset['frp'].sum())
                    local_max = float(subset['frp'].max())
                    max_frp = local_max if np.isnan(max_frp) else max(max_frp, local_max)

                    unique_hours = subset['hour_utc'].unique().tolist()
                    matched_hours.update(unique_hours)

                    local_first = subset['hour_utc'].min()
                    local_last = subset['hour_utc'].max()
                    first_match = local_first if first_match is None or local_first < first_match else first_match
                    last_match = local_last if last_match is None or local_last > last_match else last_match

        current_day += timedelta(days=1)

    overlap_rows.append(
        {
            'fire_id': row.fire_id,
            'viirs_points_in_bbox_active_window': point_count,
            'viirs_hours_in_bbox_active_window': len(matched_hours),
            'viirs_total_frp_in_bbox_active_window': total_frp,
            'viirs_max_frp_in_bbox_active_window': max_frp,
            'first_viirs_hour_utc': first_match,
            'last_viirs_hour_utc': last_match,
        }
    )

fire_overlap_df = pd.DataFrame(overlap_rows)
fire_overlap_df['first_viirs_hour_utc'] = pd.to_datetime(fire_overlap_df['first_viirs_hour_utc'], utc=True)
fire_overlap_df['last_viirs_hour_utc'] = pd.to_datetime(fire_overlap_df['last_viirs_hour_utc'], utc=True)

fire_viirs_df = fire_df.merge(fire_overlap_df, on='fire_id', how='left')
fire_viirs_df['has_viirs_in_bbox'] = fire_viirs_df['viirs_points_in_bbox_active_window'].fillna(0) > 0

fire_viirs_df['hours_to_first_viirs'] = (
    (fire_viirs_df['first_viirs_hour_utc'] - fire_viirs_df['start_hour_utc']).dt.total_seconds() / 3600.0
)
fire_viirs_df['hours_from_last_viirs_to_fire_end'] = (
    (fire_viirs_df['end_hour_utc'] - fire_viirs_df['last_viirs_hour_utc']).dt.total_seconds() / 3600.0
)

fire_overlap_summary = pd.Series(
    {
        'fires_total': int(len(fire_viirs_df)),
        'fires_with_viirs_bbox_overlap': int(fire_viirs_df['has_viirs_in_bbox'].sum()),
        'fires_without_viirs_bbox_overlap': int((~fire_viirs_df['has_viirs_in_bbox']).sum()),
        'median_viirs_points_per_fire': float(fire_viirs_df['viirs_points_in_bbox_active_window'].median()),
        'median_viirs_hours_per_fire': float(fire_viirs_df['viirs_hours_in_bbox_active_window'].median()),
        'median_hours_to_first_viirs': float(fire_viirs_df['hours_to_first_viirs'].dropna().median()),
        'median_hours_from_last_viirs_to_end': float(fire_viirs_df['hours_from_last_viirs_to_fire_end'].dropna().median()),
    },
    name='value',
)

fire_overlap_summary.to_frame()


In [ ]:
# Fire-level overlap diagnostics and edge examples
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))

axes[0].hist(np.log1p(fire_viirs_df['viirs_points_in_bbox_active_window'].fillna(0)), bins=40, color='#1f77b4')
axes[0].set_title('log1p(VIIRS points) per fire window')
axes[0].set_xlabel('log1p(points)')

axes[1].scatter(
    fire_viirs_df['size_acres'],
    fire_viirs_df['viirs_points_in_bbox_active_window'].fillna(0),
    s=16,
    alpha=0.5,
    color='#ff7f0e',
)
axes[1].set_title('Fire size vs VIIRS point count')
axes[1].set_xlabel('Fire size (acres)')
axes[1].set_ylabel('VIIRS points in bbox window')

valid_start_lag = fire_viirs_df['hours_to_first_viirs'].dropna()
axes[2].hist(valid_start_lag, bins=40, color='#2ca02c')
axes[2].set_title('Hours from fire start to first matched VIIRS hour')
axes[2].set_xlabel('Hours')

plt.tight_layout()
plt.show()

no_overlap_examples = fire_viirs_df.loc[
    ~fire_viirs_df['has_viirs_in_bbox'],
    ['fire_id', 'incident_name', 'state', 'start_time_utc', 'end_time_utc', 'size_acres'],
].sort_values(['start_time_utc', 'fire_id'])

top_overlap_examples = fire_viirs_df.loc[
    fire_viirs_df['has_viirs_in_bbox'],
    ['fire_id', 'incident_name', 'state', 'size_acres', 'viirs_points_in_bbox_active_window', 'viirs_total_frp_in_bbox_active_window', 'hours_to_first_viirs', 'hours_from_last_viirs_to_fire_end'],
].sort_values('viirs_points_in_bbox_active_window', ascending=False)

print('Fires without any matched VIIRS points (first 15 rows):')
no_overlap_examples.head(15)


## Sample-Level Match: Wildfire Index Rows vs VIIRS Source Hours

This section aligns VIIRS points to wildfire dataset samples using each sample bbox and `source_hours_utc` (daily 4-hour cadence in this run).


In [ ]:
# Load wildfire sample index (parquet preferred, checkpoint fallback)
wildfire_index_df = load_wildfire_index_df(cfg.paths.wildfire_index_output).copy()

required_index_cols = {
    'sample_id',
    'fire_id',
    'run_time_utc',
    'run_date',
    'source_hours_utc',
    'bbox_min_lon',
    'bbox_min_lat',
    'bbox_max_lon',
    'bbox_max_lat',
    'split',
}
missing_index_cols = required_index_cols.difference(wildfire_index_df.columns)
if missing_index_cols:
    raise ValueError(f'Wildfire index missing columns: {sorted(missing_index_cols)}')

wildfire_index_df['run_time_utc'] = pd.to_datetime(wildfire_index_df['run_time_utc'], utc=True, errors='coerce')
wildfire_index_df['run_date'] = pd.to_datetime(wildfire_index_df['run_date'], errors='coerce').dt.date

wildfire_index_df = wildfire_index_df.dropna(subset=['sample_id', 'run_date']).copy()
wildfire_index_df['sample_id'] = wildfire_index_df['sample_id'].astype(int)
wildfire_index_df['source_hours_list'] = wildfire_index_df['source_hours_utc'].apply(parse_source_hours)

index_overview = pd.Series(
    {
        'index_rows': int(len(wildfire_index_df)),
        'unique_fires_in_index': int(wildfire_index_df['fire_id'].nunique()),
        'run_date_min': wildfire_index_df['run_date'].min(),
        'run_date_max': wildfire_index_df['run_date'].max(),
        'aggregation_modes': wildfire_index_df.get('aggregation_mode', pd.Series(dtype=object)).dropna().nunique(),
    },
    name='value',
)

index_overview.to_frame()


In [ ]:
# Compute sample-level VIIRS overlap at sample source hours and sample bbox
viirs_by_hour = {
    hour: grp[['latitude', 'longitude', 'frp']]
    for hour, grp in viirs_df.groupby('hour_utc', sort=False)
}

sample_overlap_rows = []
for row in wildfire_index_df.itertuples(index=False):
    hours = list(row.source_hours_list)
    if not hours:
        if pd.notna(row.run_time_utc):
            hours = [int(pd.Timestamp(row.run_time_utc).hour)]
        else:
            hours = []

    points = 0
    total_frp = 0.0
    max_frp = np.nan
    matched_hour_count = 0

    for hr in hours:
        ts = pd.Timestamp(f'{row.run_date} {hr:02d}:00:00', tz='UTC')
        hour_df = viirs_by_hour.get(ts)
        if hour_df is None or hour_df.empty:
            continue

        in_bbox = (
            hour_df['longitude'].between(row.bbox_min_lon, row.bbox_max_lon)
            & hour_df['latitude'].between(row.bbox_min_lat, row.bbox_max_lat)
        )
        subset = hour_df.loc[in_bbox]
        if subset.empty:
            continue

        matched_hour_count += 1
        points += int(len(subset))
        total_frp += float(subset['frp'].sum())
        local_max = float(subset['frp'].max())
        max_frp = local_max if np.isnan(max_frp) else max(max_frp, local_max)

    sample_overlap_rows.append(
        {
            'sample_id': int(row.sample_id),
            'viirs_points_in_sample_hours_bbox': points,
            'viirs_matched_source_hours': matched_hour_count,
            'viirs_total_frp_in_sample_hours_bbox': total_frp,
            'viirs_max_frp_in_sample_hours_bbox': max_frp,
            'sample_has_viirs': points > 0,
        }
    )

sample_overlap_df = pd.DataFrame(sample_overlap_rows)
sample_viirs_df = wildfire_index_df.merge(sample_overlap_df, on='sample_id', how='left')

sample_summary = pd.Series(
    {
        'samples_total': int(len(sample_viirs_df)),
        'samples_with_viirs': int(sample_viirs_df['sample_has_viirs'].fillna(False).sum()),
        'samples_without_viirs': int((~sample_viirs_df['sample_has_viirs'].fillna(False)).sum()),
        'pct_samples_with_viirs': float(sample_viirs_df['sample_has_viirs'].fillna(False).mean()),
        'median_viirs_points_per_sample': float(sample_viirs_df['viirs_points_in_sample_hours_bbox'].fillna(0).median()),
        'median_matched_source_hours': float(sample_viirs_df['viirs_matched_source_hours'].fillna(0).median()),
    },
    name='value',
)

sample_summary.to_frame()


In [ ]:
# Sample-level overlap diagnostics by split and year
sample_viirs_df['run_year'] = pd.to_datetime(sample_viirs_df['run_date'], errors='coerce').dt.year

split_summary = (
    sample_viirs_df.groupby('split', as_index=False)
    .agg(
        samples=('sample_id', 'size'),
        samples_with_viirs=('sample_has_viirs', 'sum'),
        pct_with_viirs=('sample_has_viirs', 'mean'),
        median_viirs_points=('viirs_points_in_sample_hours_bbox', 'median'),
        median_matched_hours=('viirs_matched_source_hours', 'median'),
    )
    .sort_values('samples', ascending=False)
    .reset_index(drop=True)
)

yearly_sample_summary = (
    sample_viirs_df.groupby('run_year', as_index=False)
    .agg(
        samples=('sample_id', 'size'),
        pct_with_viirs=('sample_has_viirs', 'mean'),
        median_viirs_points=('viirs_points_in_sample_hours_bbox', 'median'),
    )
    .sort_values('run_year')
    .reset_index(drop=True)
)

fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))

split_plot = split_summary.sort_values('pct_with_viirs', ascending=True)
axes[0].barh(split_plot['split'], split_plot['pct_with_viirs'], color='#1f77b4')
axes[0].set_xlim(0, 1)
axes[0].set_title('Share of Samples with VIIRS by Split')
axes[0].set_xlabel('Fraction')

axes[1].plot(yearly_sample_summary['run_year'], yearly_sample_summary['pct_with_viirs'], marker='o', color='#ff7f0e')
axes[1].set_ylim(0, 1)
axes[1].set_title('Sample VIIRS Coverage by Run Year')
axes[1].set_xlabel('Run year')
axes[1].set_ylabel('Fraction with VIIRS')

axes[2].hist(np.log1p(sample_viirs_df['viirs_points_in_sample_hours_bbox'].fillna(0)), bins=40, color='#2ca02c')
axes[2].set_title('log1p(VIIRS points) per sample')
axes[2].set_xlabel('log1p(points)')

plt.tight_layout()
plt.show()

split_summary


In [ ]:
# Consolidated findings snapshot
findings = pd.Series(
    {
        'fires_filtered': int(len(fire_df)),
        'fires_fully_temporally_covered_by_viirs': int(fire_df['window_fully_covered_by_viirs'].sum()),
        'fires_with_bbox_time_viirs_points': int(fire_viirs_df['has_viirs_in_bbox'].sum()),
        'viirs_start_hour_utc': viirs_df['hour_utc'].min(),
        'viirs_end_hour_utc': viirs_df['hour_utc'].max(),
        'sample_rows': int(len(sample_viirs_df)),
        'sample_rows_with_viirs': int(sample_viirs_df['sample_has_viirs'].sum()),
        'median_hours_to_first_viirs_in_fire_window': float(fire_viirs_df['hours_to_first_viirs'].dropna().median()),
        'median_hours_from_last_viirs_to_fire_end': float(fire_viirs_df['hours_from_last_viirs_to_fire_end'].dropna().median()),
    },
    name='value',
)

findings.to_frame()


## Next Steps

- Use the fire/sample overlap tables here to identify candidate slices for model experiments (high-coverage vs sparse-coverage fires).
- Add exact polygon-based spatial matching as a sensitivity check if bbox overlap appears too permissive.
- Build the separate PM2.5-focused notebook next and link it to the same fire/sample IDs for cross-dataset diagnostics.
